# Tutorial 4: KL Bound

Explore the KL quadratic approximation and its cubic remainder bound.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "experiments").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ghosts.theory import (
    softmax, computeAttentionKL,
    computeVariance, computeSlopeSpread,
    klRemainderBound, verifyBound,
)

alpha0 = softmax(np.array([[1.0, 2.0, 0.5]]))
a = np.array([[0.3, -0.5, 0.8]])
var0 = computeVariance(alpha0, a)
delta = computeSlopeSpread(a)

taus = np.linspace(0, 1.5, 200)
kl_exact = [computeAttentionKL(alpha0, a, t) for t in taus]
kl_quad = [0.5 * t**2 * var0 for t in taus]
bound = [klRemainderBound(t, delta) for t in taus]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(taus, kl_exact, 'k-', lw=2, label='Exact KL')
ax.plot(taus, kl_quad, 'r--', lw=1.5, label=r'Quadratic $\frac{\tau^2}{2}$Var')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('KL divergence')
ax.set_title('KL vs quadratic approximation')
ax.legend()

ax = axes[1]
remainder = [abs(e - q) for e, q in zip(kl_exact, kl_quad)]
ax.plot(taus, remainder, 'k-', lw=2, label='|Remainder|')
ax.plot(taus, bound, 'r--', lw=1.5, label=r'Bound $C|\tau|^3\Delta^3$')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Magnitude')
ax.set_title('Remainder vs sharp bound')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
rng = np.random.default_rng(42)
ratios = []
for _ in range(100):
    n = rng.integers(2, 8)
    z = rng.standard_normal((1, n))
    a_vec = rng.standard_normal((1, n))
    tau = rng.uniform(0.01, 0.5)
    result = verifyBound(softmax(z), a_vec, tau)
    ratios.append(result['ratio'])
    assert result['valid'], f"Bound violated! ratio={result['ratio']}"

print(f"All 100 tests passed.")
print(f"Tightness ratios: min={min(ratios):.4f}, max={max(ratios):.4f}, mean={np.mean(ratios):.4f}")


In [ ]:
p = (3 + np.sqrt(3)) / 6
alpha0 = np.array([[p, 1 - p]])
a = np.array([[0.0, 1.0]])

taus = np.linspace(0.01, 0.3, 50)
ratios = []
for t in taus:
    r = verifyBound(alpha0, a, t)
    ratios.append(r['ratio'])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(taus, ratios, 'k-', lw=2)
ax.axhline(1.0, color='red', ls='--', alpha=0.5, label='Bound = 1')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Remainder / Bound')
ax.set_title(f'Tightness at Bernoulli extremum (p={p:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Max ratio (tightness): {max(ratios):.4f}")
